#1. IMPOTAÇÃO DAS BIBLIOTECAS

In [9]:

"""
============================================================

OBJETIVO:
Importar todas as bibliotecas utilizadas.

BIBLIOTECAS:

pandas / numpy:
- manipulação dos dados

matplotlib / seaborn:
- visualizações

scikit-learn:
- Machine Learning
- pré-processamento
- métricas
- pipeline
- validação

shap:
- interpretabilidade moderna (XAI)

============================================================
"""

# Manipulação de dados
import pandas as pd
import numpy as np

# Visualização
import matplotlib.pyplot as plt
import seaborn as sns

# Pipeline
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

# Pré-processamento
from sklearn.preprocessing import (
    OneHotEncoder,
    MinMaxScaler,
    label_binarize
)

# Tratamento de missing values
from sklearn.impute import SimpleImputer

# Modelo
from sklearn.linear_model import LogisticRegression

# Seleção de features
from sklearn.feature_selection import (
    SelectKBest,
    mutual_info_classif
)

# Grid Search / Validação
from sklearn.model_selection import (
    StratifiedKFold,
    GridSearchCV
)

# Balanceamento
from sklearn.utils.class_weight import (
    compute_sample_weight
)

# Métricas
from sklearn.metrics import (

    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    roc_auc_score,
    top_k_accuracy_score
)

# Calibration
from sklearn.calibration import calibration_curve

# PCA / t-SNE
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

# Explainable AI
import shap

# Utilitário
from sklearn.utils.validation import check_is_fitted

# Configuração visual
sns.set_style("whitegrid")
pd.set_option("display.max_columns", None)

# 2. CARREGAMENTO DOS DADOS

In [10]:
"""
============================================================

OBJETIVO:
Carregar os datasets de treino e teste.

ARQUIVOS:
- datasense_full_train_1sec.parquet
- datasense_full_test_1sec.parquet

IMPORTANTE:
Cada linha representa:
- uma janela temporal
- contendo estatísticas de rede
- relacionadas ao tráfego IIoT

============================================================
"""

df_train = pd.read_parquet(
    "datasense_full_train_1sec.parquet"
)

df_test = pd.read_parquet(
    "datasense_full_test_1sec.parquet"
)

print("Train shape:", df_train.shape)
print("Test shape :", df_test.shape)

Train shape: (170393, 115)
Test shape : (56798, 115)


#3. DEFINIÇÃO DO TARGET

In [11]:
"""
============================================================

OBJETIVO:
Definir o problema multiclasse.

TARGET:
label4

label4:
- contém os cenários completos de ataques
- representa 84 classes distintas

EXEMPLOS:
- benign
- dos_syn-flood
- ddos_udp-flood
- reconnaissance_portscan

============================================================
"""

target_col = "label4"

y_train = df_train[target_col]
y_test = df_test[target_col]

print("Quantidade de classes:")
print(y_train.nunique())

print("\nDistribuição das classes:")
display(y_train.value_counts().head(20))

Quantidade de classes:
84

Distribuição das classes:


label4
benign                             102600
mitm_arp-spoofing                    3148
malware_mirai-udp-flood              3006
recon_os-scan                        2969
recon_vuln-scan                      2956
recon_host-disc-tcp-ack-ping         2952
recon_host-disc-tcp-syn-ping         2945
recon_port-scan                      2939
recon_host-disc-tcp-syn-stealth      2933
recon_host-disc-arp-ping             2927
recon_host-disc-udp-ping             2919
malware_mirai-syn-flood              2647
mitm_ip-spoofing                     2280
recon_ping-sweep                     1663
dos_udp-flood                        1292
dos_icmp-flood                       1289
dos_icmp-frag-flood                  1202
ddos_icmp-flood                      1199
ddos_icmp-frag-flood                 1197
ddos_udp-frag-flood                  1197
Name: count, dtype: int64

#4. REMOÇÃO DE COLUNAS

In [12]:
"""
============================================================

OBJETIVO:
Remover apenas labels auxiliares.

IMPORTANTE:
Nesta versão:
device_name será mantido.

ABORDAGEM:
Context-Aware IDS

============================================================
"""

cols_to_drop = [

    "label1",
    "label2",
    "label3",
    "label4",
    "label_extended"
]

# MANTÉM device_name
# MANTÉM device_type

X_train = df_train.drop(columns=cols_to_drop)
X_test = df_test.drop(columns=cols_to_drop)

print("Quantidade de features:")
print(X_train.shape[1])

print("\nFeatures relacionadas aos dispositivos:")

display(

    [col for col in X_train.columns

     if "device" in col.lower()]
)

Quantidade de features:
110

Features relacionadas aos dispositivos:


['device_name', 'device_type']

#5. IDENTIFICAÇÃO DAS FEATURES

In [13]:
"""
============================================================

OBJETIVO:
Separar:
- features numéricas
- features categóricas

============================================================
"""

categorical_features = X_train.select_dtypes(

    include=["object", "category", "bool"]

).columns.tolist()

numerical_features = X_train.select_dtypes(

    include=["int64", "float64"]

).columns.tolist()

print("Features numéricas:")
print(len(numerical_features))

print("\nFeatures categóricas:")
print(len(categorical_features))

Features numéricas:
64

Features categóricas:
46


#6. PIPELINE DE PRÉ-PROCESSAMENTO

In [14]:
"""
============================================================

PIPELINE DE PRÉ-PROCESSAMENTO

OBJETIVO:
Criar pipeline organizada para preparação dos dados.

ETAPAS:
1. Tratamento de valores ausentes
2. Normalização de atributos numéricos
3. Codificação de atributos categóricos

IMPORTANTE:
- Dados numéricos e categóricos possuem
  tratamentos diferentes
- Em ambientes IIoT/OT, colunas como:
    * device
    * sensor
    * protocol
    * equipment_type
  podem ajudar na identificação de ataques

============================================================
"""

# ==========================================================
# PIPELINE NUMÉRICA
# ==========================================================
#
# Responsável por:
# - tratar valores ausentes
# - normalizar atributos numéricos
#
# Exemplo:
# packet_count
# bytes
# duration
# flow_rate
#
# ==========================================================

numeric_transformer = Pipeline([

    (
        "imputer",

        # Substitui valores ausentes
        # pela mediana da coluna
        SimpleImputer(
            strategy="median"
        )
    ),

    (
        "scaler",

        # Normaliza valores
        # para intervalo entre 0 e 1
        MinMaxScaler()
    )
])

# ==========================================================
# PIPELINE CATEGÓRICA
# ==========================================================
#
# Responsável por:
# - tratar valores ausentes
# - converter categorias em vetores binários
#
# Exemplo:
# device
# sensor
# protocol
#
# ==========================================================

categorical_transformer = Pipeline([

    (
        "imputer",

        # Substitui valores ausentes
        # pelo valor mais frequente
        SimpleImputer(
            strategy="most_frequent"
        )
    ),

    (
        "onehot",

        # Converte categorias
        # em representação numérica
        #
        # Exemplo:
        # sensor_A -> [1,0,0]
        # sensor_B -> [0,1,0]
        #
        OneHotEncoder(
            handle_unknown="ignore"
        )
    )
])

# ==========================================================
# COLUMN TRANSFORMER
# ==========================================================
#
# Une os pipelines:
# - numérico
# - categórico
#
# aplicando cada transformação
# somente nas colunas corretas
#
# ==========================================================

preprocessor = ColumnTransformer([

    (
        "num",

        numeric_transformer,

        numerical_features
    ),

    (
        "cat",

        categorical_transformer,

        categorical_features
    )
])

print("Pipeline de pré-processamento criada com sucesso.")

Pipeline de pré-processamento criada com sucesso.


#7. TRANSFORMAÇÃO DOS DADOS

In [15]:
"""
============================================================

OBJETIVO:
Aplicar preprocessamento.

============================================================
"""

X_train_processed = preprocessor.fit_transform(
    X_train
)

X_test_processed = preprocessor.transform(
    X_test
)

print("Shape após preprocessamento:")
print(X_train_processed.shape)

ValueError: Cannot cast object dtype to float64

#8. FEATURE SELECTION

In [ ]:
"""
============================================================

OBJETIVO:
Selecionar melhores features.

MÉTODO:
Mutual Information

JUSTIFICATIVA:
- funciona com variáveis contínuas
- detecta relações não-lineares
- muito usado em IDS

============================================================
"""

selector = SelectKBest(

    score_func=mutual_info_classif,

    k=120
)

X_train_selected = selector.fit_transform(

    X_train_processed,

    y_train
)

X_test_selected = selector.transform(
    X_test_processed
)

print("Quantidade final de features:")
print(X_train_selected.shape[1])

#9. NÁLISE DAS FEATURES

In [ ]:
"""
============================================================

OBJETIVO:
Analisar:
- features selecionadas
- importância estatística

============================================================
"""

feature_names = (
    preprocessor.get_feature_names_out()
)

selected_mask = selector.get_support()

selected_features = feature_names[
    selected_mask
]

scores = selector.scores_[selected_mask]

feature_importance = pd.DataFrame({

    "Feature": selected_features,

    "Mutual_Information": scores
})

feature_importance = feature_importance.sort_values(

    by="Mutual_Information",

    ascending=False
)

display(feature_importance.head(20))

#10. VISUALIZAÇÃO DAS FEATURES

In [ ]:
"""
============================================================

OBJETIVO:
Visualizar top features.

============================================================
"""

plt.figure(figsize=(10,6))

sns.barplot(

    data=feature_importance.head(15),

    x="Mutual_Information",

    y="Feature"
)

plt.title(
    "Top 15 Features Mais Relevantes"
)

plt.show()

#11. SAMPLE WEIGHT

In [ ]:
"""
============================================================

OBJETIVO:
Balancear automaticamente as classes.

============================================================
"""

sample_weight = compute_sample_weight(

    class_weight="balanced",

    y=y_train
)

print("Sample weights calculados.")

#12. DEFINIÇÃO DO MODELO

In [ ]:
"""
============================================================

OBJETIVO:
Criar Logistic Regression Multinomial.

============================================================
"""

model = LogisticRegression(

    multi_class="multinomial",

    max_iter=3000,

    C=1.0,

    tol=1e-3,

    random_state=42
)

#13. GRID SEARCH

In [ ]:
"""
============================================================

OBJETIVO:
Buscar melhores hiperparâmetros.

MÉTRICA:
F1 Macro

============================================================
"""

param_grid = {

    "C": [0.3, 0.7, 1.0]
}

cv = StratifiedKFold(

    n_splits=3,

    shuffle=True,

    random_state=42
)

grid = GridSearchCV(

    estimator=model,

    param_grid=param_grid,

    scoring="f1_macro",

    cv=cv,

    n_jobs=-1,

    verbose=2,

    return_train_score=True
)

grid.fit(

    X_train_selected,

    y_train,

    sample_weight=sample_weight
)

print("Melhores parâmetros:")
print(grid.best_params_)

print("\nMelhor F1 Macro:")
print(grid.best_score_)

#14. PREDIÇÃO MULTICLASSE

In [ ]:
"""
============================================================

OBJETIVO:
Executar:
- predições
- probabilidades
- análise de confiança

============================================================
"""

best_model = grid.best_estimator_

check_is_fitted(best_model)

# probabilidades
y_prob = best_model.predict_proba(
    X_test_selected
)

# predição final
y_pred = best_model.predict(
    X_test_selected
)

# classes
classes = best_model.classes_

# maior probabilidade
max_prob = np.max(y_prob, axis=1)

# segunda maior probabilidade
sorted_probs = np.sort(y_prob, axis=1)

second_best_prob = sorted_probs[:, -2]

# margem de confiança
confidence_margin = (

    max_prob -

    second_best_prob
)

# dataframe final
df_result = pd.DataFrame({

    "Classe_Real": y_test.values,

    "Classe_Predita": y_pred,

    "Probabilidade_Maxima": max_prob,

    "Segunda_Probabilidade": second_best_prob,

    "Margem_Confianca": confidence_margin
})

# correto/incorreto
df_result["Correto"] = (

    df_result["Classe_Real"]

    ==

    df_result["Classe_Predita"]
)

print("Quantidade total:")
print(len(df_result))

print("\nPredições corretas:")
print(df_result["Correto"].sum())

print("\nPredições incorretas:")
print((~df_result["Correto"]).sum())

display(df_result.head())

#15. AMOSTRAS MAIS CONFIANTES

In [ ]:
"""
============================================================

OBJETIVO:
Mostrar amostras com maior confiança.

============================================================
"""

mais_confiantes = df_result.sort_values(

    by="Probabilidade_Maxima",

    ascending=False
)

display(
    mais_confiantes.head(20)
)

#16. AMOSTRAS MAIS INCERTAS

In [ ]:
"""
============================================================

OBJETIVO:
Mostrar amostras mais difíceis.

============================================================
"""

mais_incertas = df_result.sort_values(

    by="Margem_Confianca",

    ascending=True
)

display(
    mais_incertas.head(20)
)

#17. MÉTRICAS

In [ ]:
"""
============================================================

OBJETIVO:
Avaliar desempenho do modelo.

============================================================
"""

print("Accuracy:")
print(
    accuracy_score(y_test, y_pred)
)

print("\nF1 Macro:")
print(
    f1_score(
        y_test,
        y_pred,
        average="macro"
    )
)

print("\nF1 Weighted:")
print(
    f1_score(
        y_test,
        y_pred,
        average="weighted"
    )
)

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred
    )
)

#18. TOP-K ACCURACY

In [ ]:
"""
============================================================

OBJETIVO:
Avaliar:
- Top-1
- Top-3
- Top-5

============================================================
"""

top1 = top_k_accuracy_score(

    y_test,

    y_prob,

    k=1,

    labels=classes
)

top3 = top_k_accuracy_score(

    y_test,

    y_prob,

    k=3,

    labels=classes
)

top5 = top_k_accuracy_score(

    y_test,

    y_prob,

    k=5,

    labels=classes
)

print("Top-1 Accuracy:", top1)
print("Top-3 Accuracy:", top3)
print("Top-5 Accuracy:", top5)

#19. ROC-AUC MULTICLASSE

In [ ]:
"""
============================================================

OBJETIVO:
ROC-AUC One-vs-Rest multiclasses.

============================================================
"""

y_test_bin = label_binarize(

    y_test,

    classes=classes
)

roc_auc_macro = roc_auc_score(

    y_test_bin,

    y_prob,

    average="macro",

    multi_class="ovr"
)

roc_auc_weighted = roc_auc_score(

    y_test_bin,

    y_prob,

    average="weighted",

    multi_class="ovr"
)

print("ROC-AUC Macro:")
print(roc_auc_macro)

print("\nROC-AUC Weighted:")
print(roc_auc_weighted)

#20. MATRIZ DE CONFUSÃO

In [ ]:
"""
============================================================

OBJETIVO:
Visualizar erros do modelo.

============================================================
"""

cm = confusion_matrix(
    y_test,
    y_pred
)

plt.figure(figsize=(15,12))

sns.heatmap(
    cm,
    cmap="Blues"
)

plt.title("Confusion Matrix")

plt.xlabel("Predito")
plt.ylabel("Real")

plt.show()

#21. CLASSES MAIS CONFUNDIDAS

In [ ]:
"""
============================================================

OBJETIVO:
Encontrar ataques mais confundidos.

============================================================
"""

confusions = []

for i in range(len(classes)):

    for j in range(len(classes)):

        if i != j:

            confusions.append({

                "Classe_Real": classes[i],

                "Classe_Predita": classes[j],

                "Quantidade": cm[i, j]
            })

conf_df = pd.DataFrame(confusions)

conf_df = conf_df.sort_values(

    by="Quantidade",

    ascending=False
)

display(conf_df.head(20))

#22. PCA

In [ ]:
"""
============================================================

OBJETIVO:
Visualização PCA.

============================================================
"""

pca = PCA(n_components=2)

X_pca = pca.fit_transform(
    X_test_selected
)

df_pca = pd.DataFrame({

    "PCA1": X_pca[:,0],

    "PCA2": X_pca[:,1],

    "Classe": y_test.values
})

top_classes = (
    y_test.value_counts()
    .head(10)
    .index
)

df_plot = df_pca[
    df_pca["Classe"].isin(top_classes)
]

plt.figure(figsize=(12,8))

sns.scatterplot(

    data=df_plot,

    x="PCA1",

    y="PCA2",

    hue="Classe",

    alpha=0.7
)

plt.title("PCA - Top 10 Classes")

plt.show()

#24. t-SNE

In [ ]:
"""
============================================================

OBJETIVO:
Visualização não-linear das classes.

============================================================
"""

sample_size = 3000

X_tsne_sample = X_test_selected[:sample_size]

y_tsne_sample = y_test.iloc[:sample_size]

tsne = TSNE(

    n_components=2,

    perplexity=30,

    random_state=42
)

X_tsne = tsne.fit_transform(
    X_tsne_sample
)

df_tsne = pd.DataFrame({

    "TSNE1": X_tsne[:,0],

    "TSNE2": X_tsne[:,1],

    "Classe": y_tsne_sample.values
})

top_classes = (
    y_test.value_counts()
    .head(10)
    .index
)

df_plot = df_tsne[
    df_tsne["Classe"].isin(top_classes)
]

plt.figure(figsize=(12,8))

sns.scatterplot(

    data=df_plot,

    x="TSNE1",

    y="TSNE2",

    hue="Classe",

    alpha=0.7
)

plt.title("t-SNE - Top 10 Classes")

plt.show()

#25. SHAP VALUES

In [ ]:
"""
============================================================

OBJETIVO:
Interpretabilidade do modelo.

============================================================
"""

sample_size = 1000

X_sample = X_test_selected[:sample_size]

explainer = shap.Explainer(

    best_model,

    X_sample
)

shap_values = explainer(X_sample)

shap.summary_plot(

    shap_values,

    X_sample,

    feature_names=selected_features
)